# Module C, Week 6: Decision Trees and Random Forests
### Integrated Capstone Project: Predicting Corporate Bankruptcy from Financial Ratios
**Astha Goel**

---

## Project context

This capstone predicts corporate bankruptcy from published financial ratios, running the same pipeline
separately on two datasets and comparing results rather than merging them:

| | Poland | Taiwan |
|---|---|---|
| Observations | 43,405 firm-years (2000 to 2013) | 6,819 firms (1999 to 2009) |
| Features | 64 financial ratios | 95 financial ratios |
| Bankrupt | 2,091 (4.8%) | 220 (3.2%) |

**Established so far.** Module B: no single ratio predicts bankruptcy alone, PCA does not compress these data
(PC1 captures only 15.3% and 13.4% of variance), yet a random forest reaches 0.9395 (Poland) and 0.9482 (Taiwan).
Week 1: severe multicollinearity (a Taiwanese pair at VIF ≈ 10¹⁵), and added flexibility helps Poland and does not help Taiwan. Week 2: features ranging to 10¹⁰ break linear models until winsorized; lasso removes redundant
ratio families automatically. Week 4: scaler choice swings Poland's ROC-AUC by 0.21, `QuantileTransformer` wins
both datasets, and `class_weight='balanced'` degrades calibration without improving ranking.

## This week

Week 6 covers decision trees and random forests. Module B already applied random forests to these datasets, so
rather than repeat that, this notebook examines **why** they work here, which turns out to connect directly to
the functional-form question running through the whole project.

Three things are worth establishing:

1. A single decision tree shows overfitting about as clearly as possible on this data, since depth can be tuned
   continuously from underfitting to memorization.
2. Ensembling corrects exactly that failure, and the mechanism is visible in the numbers.
3. Trees capture interactions natively, which is why they dominate in Poland and merely match in Taiwan.

## 1. Setup and loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

# rank-based scaling, selected in Week 4 after comparing six scalers
qt = lambda: QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE)

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV

In [ ]:
# ── TAIWAN ────────────────────────────────────────────────────────────────────
df_taiwan = pd.read_csv('Taiwan_data_capstone.csv')
df_taiwan.columns = df_taiwan.columns.str.strip()   # headers carry a leading space
df_taiwan['Bankrupt?'] = pd.to_numeric(df_taiwan['Bankrupt?'], errors='coerce').astype(int)

numeric_cols_taiwan = [c for c in df_taiwan.select_dtypes(include='number').columns
                       if c != 'Bankrupt?']
for col in numeric_cols_taiwan:
    df_taiwan[col] = df_taiwan[col].fillna(df_taiwan[col].median())
numeric_cols_taiwan = [c for c in numeric_cols_taiwan if df_taiwan[c].nunique() > 1]

print(f"Taiwan: {df_taiwan.shape}, {len(numeric_cols_taiwan)} usable features")
print(df_taiwan['Bankrupt?'].value_counts().to_dict())

In [ ]:
# ── POLAND ────────────────────────────────────────────────────────────────────
df_polish = pd.read_csv('Polish_dataset_capstone.csv')

column_names_64 = [
    'net_profit_to_total_assets', 'total_liabilities_to_total_assets',
    'working_capital_to_total_assets', 'current_assets_to_short_term_liabilities',
    'cash_to_operating_expenses', 'retained_earnings_to_total_assets',
    'EBIT_to_total_assets', 'book_value_of_equity_to_total_liabilities',
    'sales_to_total_assets', 'equity_to_total_assets',
    'gross_profit_to_short_term_liabilities', 'gross_profit_to_total_assets',
    'gross_profit_to_sales', 'gross_profit_to_total_assets_2',
    'working_capital_to_fixed_assets', 'receivables_to_total_assets',
    'liabilities_to_total_assets_duration', 'EBIT_to_total_assets_2',
    'sales_to_total_assets_2', 'working_capital_to_sales',
    'sales_to_inventories', 'net_profit_to_sales',
    'gross_profit_to_total_assets_3', 'current_liabilities_to_equity',
    'short_term_liabilities_to_equity', 'working_capital',
    'depreciation_to_total_assets', 'inventories_to_sales',
    'net_profit_to_inventories', 'total_liabilities_to_equity',
    'profit_on_sales_to_total_assets', 'total_liabilities_to_cash',
    'operating_expenses_to_short_term_liabilities', 'operating_expenses_to_total_liabilities',
    'profit_to_total_assets', 'net_profit_to_total_assets_2',
    'total_liabilities_to_operating_cash', 'total_liabilities_to_total_assets_rotation',
    'profit_to_long_term_liabilities', 'total_assets_to_total_liabilities',
    'short_term_liabilities_to_long_term_liabilities', 'short_term_liabilities_to_equity_2',
    'long_term_liabilities_to_equity', 'sales_to_short_term_liabilities',
    'inventory_to_receivables', 'receivables_to_sales',
    'net_profit_to_total_assets_3', 'total_costs_to_total_sales',
    'long_term_liabilities_to_total_assets', 'sales_to_equity',
    'sales_to_total_assets_3', 'liabilities_to_working_capital',
    'operating_cash_to_liabilities', 'operating_cash_to_total_liabilities',
    'operating_cash_to_sales', 'sales_to_receivables',
    'operating_expenses_to_sales', 'total_costs_to_total_assets',
    'profit_on_operating_activities', 'depreciation_to_profit',
    'total_liabilities_to_profit', 'profit_on_operating_activities_to_financial_expenses',
    'working_capital_to_liabilities', 'equity_to_fixed_assets'
]
df_polish = df_polish.rename(columns={f'A{i+1}': column_names_64[i] for i in range(64)})

# force ratio columns numeric, else select_dtypes silently skips any that arrive as strings
for c in column_names_64:
    if c in df_polish.columns:
        df_polish[c] = pd.to_numeric(df_polish[c], errors='coerce')
df_polish['class'] = pd.to_numeric(df_polish['class'], errors='coerce').astype(int)

numeric_cols_polish = [c for c in df_polish.select_dtypes(include='number').columns
                       if c not in ['class', 'year', 'id']]
for col in numeric_cols_polish:
    df_polish[col] = df_polish[col].fillna(df_polish[col].median())
numeric_cols_polish = [c for c in numeric_cols_polish if df_polish[c].nunique() > 1]

print(f"Poland: {df_polish.shape}, {len(numeric_cols_polish)} usable features")
print(df_polish['class'].value_counts().to_dict())

## 2. A single decision tree, and how it overfits

A decision tree splits the data repeatedly on one feature at a time, choosing at each step the split that best
separates the classes. Depth directly controls how much the tree can memorize: a shallow tree can express only simple rules, while an
unrestricted tree keeps splitting until every leaf is pure: perfectly classifying the training data by
memorizing it.

Trees need no feature scaling at all, since they split on rank order. That invariance is precisely why the
10¹⁰-magnitude corrupted values documented in Week 2 passed unnoticed through all of Module B.

Reporting train and test scores together at each depth makes the overfitting visible rather than inferred.

In [ ]:
def depth_sweep(df, feats, target, name, depths=(2, 5, 10, 20, None)):
    X, y = df[feats], df[target].values
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    rows = []
    for d in depths:
        tree = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(Xtr, ytr)
        tr = roc_auc_score(ytr, tree.predict_proba(Xtr)[:, 1])
        te = roc_auc_score(yte, tree.predict_proba(Xte)[:, 1])
        rows.append({'max_depth': str(d), 'leaves': tree.get_n_leaves(),
                     'train ROC-AUC': tr, 'test ROC-AUC': te, 'gap': tr - te})
    res = pd.DataFrame(rows)
    print(f"\n=== {name}: single decision tree, depth sweep ===")
    print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    best = res.loc[res['test ROC-AUC'].idxmax()]
    print(f"   Best test performance at depth {best['max_depth']} "
          f"({best['test ROC-AUC']:.4f}); unrestricted reaches train "
          f"{res['train ROC-AUC'].iloc[-1]:.4f} but test {res['test ROC-AUC'].iloc[-1]:.4f}")
    return res

depth_pl = depth_sweep(df_polish, numeric_cols_polish, 'class', 'POLAND')
depth_tw = depth_sweep(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (res, name) in zip(axes, [(depth_pl, 'Poland'), (depth_tw, 'Taiwan')]):
    x = np.arange(len(res))
    ax.plot(x, res['train ROC-AUC'], marker='o', ms=7, color='indianred', label='train')
    ax.plot(x, res['test ROC-AUC'], marker='s', ms=7, color='steelblue', label='test')
    ax.fill_between(x, res['test ROC-AUC'], res['train ROC-AUC'],
                    color='indianred', alpha=0.12)
    ax.set_xticks(x); ax.set_xticklabels(res['max_depth'])
    ax.set_xlabel('max_depth'); ax.set_ylabel('ROC-AUC'); ax.set_ylim(0.45, 1.03)
    ax.set_title(f'{name}: shaded area = overfitting gap', fontsize=10)
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

### Textbook overfitting, visible directly

Both datasets reach a training ROC-AUC of exactly **1.000** when the tree is unrestricted: perfect memorization, while test performance falls well below its peak. The optimum sits at a depth of about five in both cases, and
every additional level past that buys memorization rather than generalization.

This is the clearest evidence of overfitting in the whole project, because the divergence is observed directly
rather than inferred from a gap between methods. It also shows why `max_depth` and `min_samples_leaf` are the
hyperparameters that matter most for trees: they control how much the model can memorize.

Below, a depth-3 tree is printed to show what the model is actually doing. The thresholds are readable financial rules, which is the one real advantage a single tree has over every other method here.

In [ ]:
shallow = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
X_tw, y_tw = df_taiwan[numeric_cols_taiwan], df_taiwan['Bankrupt?'].values
shallow.fit(X_tw, y_tw)

fig, ax = plt.subplots(figsize=(15, 6))
plot_tree(shallow, feature_names=[c[:26] for c in numeric_cols_taiwan],
          class_names=['solvent', 'bankrupt'], filled=True, rounded=True,
          fontsize=7, impurity=False, ax=ax)
ax.set_title('Taiwan: a depth-3 tree is readable as financial rules', fontsize=11)
plt.tight_layout(); plt.show()

## 3. Random forests: why averaging fixes it

A random forest fits many trees and averages them. Two sources of randomness keep the trees from being identical,
which is what makes averaging work:

- **Bagging**: each tree trains on a bootstrap sample of the rows
- **Feature subsampling**: each split considers only a random subset of features (`max_features`)

Individually the trees still overfit. But because they overfit *differently*, their errors are partly independent
and averaging cancels much of the variance while keeping the signal. That is the whole mechanism, and it predicts
something testable: the forest should beat the best single tree by a wide margin.

In [ ]:
def forest_vs_tree(df, feats, target, name):
    X, y = df[feats], df[target].values
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    best_tree_auc, best_depth = -1, None
    for d in (2, 5, 10, 20, None):
        t = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(Xtr, ytr)
        a = roc_auc_score(yte, t.predict_proba(Xte)[:, 1])
        if a > best_tree_auc:
            best_tree_auc, best_depth = a, d

    rf = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=5,
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
    rf_auc = roc_auc_score(yte, rf.predict_proba(Xte)[:, 1])
    rf_train = roc_auc_score(ytr, rf.predict_proba(Xtr)[:, 1])

    logit = make_pipeline(qt(), LogisticRegression(max_iter=3000)).fit(Xtr, ytr)
    logit_auc = roc_auc_score(yte, logit.predict_proba(Xte)[:, 1])

    print(f"\n=== {name} ===")
    print(f"   Best single tree (depth {best_depth}): {best_tree_auc:.4f}")
    print(f"   Random forest (200 trees):     {rf_auc:.4f}  (train {rf_train:.4f})")
    print(f"   Logistic regression (Week 4):  {logit_auc:.4f}")
    print(f"   Forest improvement over tree:  {rf_auc - best_tree_auc:+.4f}")
    print(f"   Forest advantage over linear:  {rf_auc - logit_auc:+.4f}")
    return {'tree': best_tree_auc, 'forest': rf_auc, 'logistic': logit_auc, 'rf_model': rf}

res_pl = forest_vs_tree(df_polish, numeric_cols_polish, 'class', 'POLAND')
res_tw = forest_vs_tree(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN')

### The forest advantage differs sharply by country

Both forests improve enormously on their best single tree, confirming the variance-reduction mechanism. But the
comparison against logistic regression splits the two datasets:

- In **Poland** the forest beats the linear model by a wide margin.
- In **Taiwan** the linear model is competitive with the forest, and in Week 4 slightly exceeded it.

This is the functional-form result again, now from a fourth independent direction. Trees capture interactions
natively, every path from root to leaf is a conjunction of conditions, which is an interaction by definition, so
they should dominate precisely where interactions matter. They do: Poland. And they should offer no advantage
where the signal is additive. They don't: Taiwan.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
labels = ['Best single tree', 'Random forest', 'Logistic regression']
x = np.arange(3); w = 0.37
ax.bar(x - w/2, [res_pl['tree'], res_pl['forest'], res_pl['logistic']], w,
       label='Poland', color='steelblue')
ax.bar(x + w/2, [res_tw['tree'], res_tw['forest'], res_tw['logistic']], w,
       label='Taiwan', color='coral')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9.5)
ax.set_ylim(0.70, 1.0); ax.set_ylabel('Test ROC-AUC')
ax.set_title('Forests help Poland far more than Taiwan', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 4. Hyperparameter tuning and what the parameters mean

The forest's main hyperparameters are limits on how much the model can memorize, and each has a specific meaning here:

- **`n_estimators`**: number of trees. More is monotonically better for accuracy, with diminishing returns;
  it cannot overfit, only cost time.
- **`max_depth`**: capacity per tree. The direct overfitting control demonstrated above.
- **`min_samples_leaf`**: minimum observations per leaf. Raising it prevents leaves built from one or two firms,
  which matters here because bankruptcies are rare and such leaves are almost pure noise.
- **`max_features`**: features considered per split. Lower values decorrelate the trees more, which is what
  makes averaging effective.

Tuning uses stratified cross-validation so each fold preserves the rare bankrupt class at its true rate, and
ROC-AUC rather than accuracy, since predicting "solvent" for everything would already score about 96%.

In [ ]:
def tune_forest(df, feats, target, name, n_sample=15000):
    d = df.sample(n_sample, random_state=RANDOM_STATE) if n_sample and n_sample < len(df) else df
    X, y = d[feats], d[target].values
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(
        RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1),
        param_grid={'max_depth': [10, 20], 'min_samples_leaf': [1, 5]},
        scoring='roc_auc', cv=cv, n_jobs=1)
    grid.fit(Xtr, ytr)

    best = grid.best_estimator_
    te = roc_auc_score(yte, best.predict_proba(Xte)[:, 1])
    print(f"\n=== {name}: tuned random forest ===")
    print(f"   Best params: {grid.best_params_}")
    print(f"   CV ROC-AUC:   {grid.best_score_:.4f}")
    print(f"   Test ROC-AUC: {te:.4f}")
    return grid, best

grid_tw, best_tw = tune_forest(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN', n_sample=None)
grid_pl, best_pl = tune_forest(df_polish, numeric_cols_polish, 'class', 'POLAND')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
for ax, (model, feats, name) in zip(axes, [(res_pl['rf_model'], numeric_cols_polish, 'Poland'),
                                           (res_tw['rf_model'], numeric_cols_taiwan, 'Taiwan')]):
    imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False).head(12)
    imp.sort_values().plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'{name}: top 12 feature importances', fontsize=10)
    ax.set_xlabel('importance'); ax.tick_params(axis='y', labelsize=7)
plt.tight_layout(); plt.show()

for name, model, feats in [('Poland', res_pl['rf_model'], numeric_cols_polish),
                           ('Taiwan', res_tw['rf_model'], numeric_cols_taiwan)]:
    imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
    print(f"{name}: top feature carries {imp.iloc[0]:.3f} of total importance; "
          f"top 10 carry {imp.head(10).sum():.3f}")

No single feature dominates either forest, the top predictor carries well under 10% of total importance, and
importance declines gradually across many features. That is reassuring, not disappointing: a single
feature carrying an outsized share would suggest leakage, meaning some column was effectively giving away the answer.
That is consistent with the Module B finding that no individual ratio predicts bankruptcy on its own.

## 5. A limitation that affects the Polish results

The Polish dataset has a panel structure: the same firm appears across multiple forecasting horizons. A random
train/test split can therefore place correlated observations of one firm on both sides, letting the model see a
near-copy of a test case during training.

Week 4 tested this on linear models and found the effect negligible. But a random forest has far more capacity to
memorize firm-specific patterns, so that test does **not** clear it. If an `id` column is available, the check
below is the one that settles the question.

In [ ]:
if 'id' in df_polish.columns:
    from sklearn.model_selection import GroupShuffleSplit

    Xp, yp = df_polish[numeric_cols_polish].values, df_polish['class'].values
    groups = df_polish['id'].values
    print(f"Rows: {len(df_polish):,}   unique firms: {df_polish['id'].nunique():,}   "
          f"mean obs/firm: {len(df_polish)/df_polish['id'].nunique():.2f}\n")

    Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(
        Xp, yp, test_size=0.2, random_state=RANDOM_STATE, stratify=yp)
    tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2,
                                            random_state=RANDOM_STATE).split(Xp, yp, groups))

    for label, (a, b, c, d) in {
        'Random split (firms may appear in both)': (Xtr_r, Xte_r, ytr_r, yte_r),
        'Group split (firms kept together)':       (Xp[tr_idx], Xp[te_idx], yp[tr_idx], yp[te_idx]),
    }.items():
        rf = RandomForestClassifier(n_estimators=150, max_depth=20, min_samples_leaf=5,
                                    random_state=RANDOM_STATE, n_jobs=-1).fit(a, c)
        print(f"   {label:42s} ROC-AUC = {roc_auc_score(d, rf.predict_proba(b)[:, 1]):.4f}")
else:
    print("No 'id' column present - firm-level grouping cannot be tested.")

### The leakage concern does not hold up

The Polish data contains 10,503 distinct firms averaging about four observations each, so the panel structure is
real and the concern was legitimate. But the group-aware split scores **higher** than the random split, not
lower. Had leakage been inflating the random-split figure, the group-aware number would have fallen; it rose.

That settles an open question carried since Module B. The Polish forest results do not need to be discounted,
and the 0.9395 reported there can stand rather than being treated as an upper bound. The likely reason leakage
is mild here is that a firm's ratios change substantially between forecasting horizons, a firm five years from
failure genuinely looks different from the same firm one year out, so the repeated observations are far less
redundant than the shared `id` suggests.

Either way it is worth reporting. A limitation that turns out not to matter is still worth testing and saying so,
because otherwise I would be carrying an untested caveat through the whole project.

## 6. Conclusions

1. **Single trees show overfitting more clearly than anything else in this project.** Both datasets reach
   a training ROC-AUC of exactly 1.000 unrestricted while test performance falls well below its depth-5 peak. The
   divergence is observed, not inferred.

2. **Ensembling works by the mechanism the theory describes.** Forests beat the best single tree by a wide margin
   in both datasets, because bagging and feature subsampling make the trees' errors partly independent so
   averaging cancels variance while preserving signal.

3. **The forest's advantage over linear models is country-specific, and that is the project's central finding
   arriving from a fourth direction.** Trees capture interactions natively, each root-to-leaf path *is* an
   interaction, so they dominate in Poland, where interactions matter, and merely match in Taiwan, where the
   signal is additive. Polynomial terms (Week 1), interaction terms (Weeks 1 and 2), kernel choice (Week 5), and
   now model family all agree.

4. **No evidence of leakage.** Importance is spread across many features in both forests rather than concentrated
   in one, which is what a leaking column would look like.

5. **The panel-leakage concern is resolved, favorably.** A group-aware split keyed on firm identity scores
   higher than a random split, so firm-level repetition was not inflating the Polish results. The figures can
   stand as reported.

**Carried forward:** Poland gets tree-based and gradient-boosted models; Taiwan keeps the interpretable logistic
model from Week 4, since it costs nothing in accuracy there. The grouped-CV caveat carried since Module B can be
retired.